In [ ]:
# imports
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import shap

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.metrics import roc_auc_score, average_precision_score, brier_score_loss
from sklearn.calibration import calibration_curve

In [ ]:
# load data
# turn true/false columns to 1/0. just helps avoid mixed-columns types
X_train = pd.read_csv("../data/processed/X_train.csv").astype(float)
X_test = pd.read_csv("../data/processed/X_test.csv").astype(float)

# 1 = died or too sick, 0 is anything else. answer we predict
y_train = pd.read_csv("../data/processed/y_train.csv")["event_adverse"]
y_test = pd.read_csv("../data/processed/y_test.csv")["event_adverse"]

# check the shape
print(X_train.shape, X_test.shape)
print("adverse rate:", round(y_train.mean(), 3), round(y_test.mean(), 3)) 

In [ ]:
# try out smaller sample of data. stratify keeps same share of adverse events in sample
# _ splits data then throws away test features and labels
X_small, _, y_small, _ = train_test_split(
    X_train, y_train, train_size=40000, stratify=y_train, random_state=42)

In [ ]:
# test each model we created
models = {
    "Logisitic Regression":(
        LogisticRegression(max_iter=2000),
        {"C": [0.1, 1, 10]} # strong regularization, defauly, weak
    ),
"Random Forest": (
        RandomForestClassifier(n_estimators=200, n_jobs=-1, random_state=42),
        {"max_depth": [6, 12],                     # how deep each tree can go
         "min_samples_leaf": [10, 50]}             # minimum patients per leaf, bigger is simpler
    ),
    "XGBoost": (
        XGBClassifier(n_estimators=300, learning_rate=0.05, n_jobs=-1, random_state=42),
        {"max_depth": [3, 6],                      # how deep each tree can go
         "min_child_weight": [1, 10]}              # bigger means more cautious
    ),
}

In [ ]:
# try every setting & keep best one for each model
best_models = {} # best version of each model
cv_scores = {} #how well best version did turing tryout

for name, (model, settings) in models.items():
    #GRID_SEARCH
    # splits sample into 3 parts, trains on 2, averages results
    search = GridSearchCV(model, settings, scoring="average_precision", cv=3)
    search.fit(X_small, y_small)
    best_models[name] = search.best_estimator_
    cv_scores[name] = search.best_score_
    print(name, "| best settings:", search.best_params_, "| PR-AUC:", round(search.best_score_, 3))

# winner is one with best tryout score. test set not used to choose
best_name = max(cv_scores, key=cv_scores.get)
print("Winner:", best_name)

# train each best model on all training data
for name, model in best_models.items():
    model.fit(X_train, y_train)


In [ ]:
# grade all 3 on test data
def top_decile_lift(y, p):
    # target at least 2.5. take the 10% of patients the model rates riskiest and how much higher is their risk adverse rate than average?
    y = np.asarray(y)
    top = np.argsort(-np.asarray(p))[: len(y) // 10]
    return y[top].mean() / y.mean()

results = {}
for name, model in best_models.items():
    p = model.predict_proba(X_test)[:, 1]    # predicted risk from 0 to 1 for each patient
    results[name] = {
        "ROC-AUC": roc_auc_score(y_test, p),            # target: 0.78 or higher
        "PR-AUC": average_precision_score(y_test, p),   # target: 0.65 or higher
        "Brier": brier_score_loss(y_test, p),           # target: 0.18 or lower (lower is better)
        "Top-decile lift": top_decile_lift(y_test, p),  # target: 2.5 or higher
    }

pd.DataFrame(results).T.round(3)

In [ ]:
# calibration: can predicted risks be trusted?
# if model says 20% risk, around 20% of patients should really have an adverse event. lines close to dotted diagonal imply trustworthiness
plt.figure(figsize=(6, 5))
plt.plot([0, 1], [0, 1], "k--", label="Perfect")

for name, model in best_models.items():
    p = model.predict_proba(X_test)[:, 1]
    actual, predicted = calibration_curve(y_test, p, n_bins=10, strategy="quantile")
    plt.plot(predicted, actual, marker="o", label=name)

plt.xlabel("Predicted risk")
plt.ylabel("Actual adverse-event rate")
plt.title("Calibration")
plt.legend()
plt.show()

In [ ]:
# shap (what drives model's winning predictions)
best_model = best_models[best_name]
X_explain = X_test.sample(1000, random_state=42)   # since SHAP is slow, only use 1000 patients

# Logistic Regression & tree models need different SHAP tools
if best_name == "Logistic Regression":
    explainer = shap.LinearExplainer(best_model, X_train.sample(1000, random_state=42))
else:
    explainer = shap.TreeExplainer(best_model)

shap_values = explainer.shap_values(X_explain)

# some models return the values in a different shape, which keeps only the "adverse event"
if isinstance(shap_values, list):
    shap_values = shap_values[1]
shap_values = np.asarray(shap_values)
if shap_values.ndim == 3:
    shap_values = shap_values[:, :, 1]

# each dot is one patient. right of 0 raises risk, left of 0 lowers it
# red = high value, blue is low value.
shap.summary_plot(shap_values, X_explain, max_display=15)

# The same information as a ranked list: bigger number = bigger effect on predictions
# bigger number = bigger effect on population
importance = pd.Series(np.abs(shap_values).mean(axis=0), index=X_explain.columns)
importance.sort_values(ascending=False).head(10).round(3)